# Music Annotation Project V8.8
## Notebook 59R1: REMOVE Deployment Fit, Calibration and Immutable Freeze

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook repairs one specific archival gap between the completed V8.7 development lineage and fresh-page operational scoring in Notebooks 60R1/60R2.

Notebook 50 selected `REMOVE = local_nonlinear` and demonstrated the REMOVE-only policy with source-group out-of-fold evidence, but it persisted only fold-specific isotonic calibrators and did not persist the fitted REMOVE estimators. Notebook 59R1 therefore:

1. resolves the exact completed Notebook 50 run;
2. resolves the exact Notebook 49 source recorded through Notebook 49R4 lineage;
3. reconstructs Notebook 50's locked pixel population and exact ordered predictor schema;
4. verifies the selected family is `local_nonlinear` and the Notebook 50 verdict authorises REMOVE-only crossover;
5. fits that already-selected estimator family once on all permitted development rows;
6. fits one final isotonic calibrator from the preserved Notebook 50 **OOF raw REMOVE scores and REMOVE truth**, never from in-sample full-fit scores;
7. saves a deployable model/calibrator/schema/policy bundle;
8. reloads every saved object and requires exact deterministic replay on a frozen multi-page sample;
9. writes a content-addressed immutable freeze manifest and curated upload package.

### Non-negotiable boundaries

- This notebook does **not** reopen model selection.
- It does **not** tune hyperparameters or thresholds.
- It does **not** operationalise `SAFE_PROTECT`.
- It does **not** use clean images or truth-derived fields as predictors.
- It does **not** claim the all-development fit reproduces Notebook 50 OOF probabilities.
- Notebook 50 OOF evidence remains the scientific selection evidence. Notebook 59R1 creates the missing deterministic fresh-page scoring object.
- Every unsupported, invalid or uncertain location remains exact M3 fallback downstream.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow scikit-learn scipy joblib

## 1. Configuration, explicit source identity and frozen contract

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, inspect, math
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB50_ROOT = ROOT/'observable_residual_mask_m5_v8_7'
OUTPUT_ROOT = ROOT/'observable_residual_mask_m5_v8_8_deployment_freeze'

# Set this only if the authoritative completed run has moved.
NB50_RUN_OVERRIDE = None

SEED = 42
SELECTED_TARGET = 'REMOVE'
EXPECTED_SELECTED_FAMILY = 'local_nonlinear'
EXPECTED_NB50_VERSION = '8.7-notebook50-asymmetric-m5'
EXPECTED_NB50_VERDICT = 'M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER'

# Exact Notebook 50 model-family definition. No tuning is permitted here.
MODEL_PARAMS = {
    'max_iter': 300,
    'max_leaf_nodes': 15,
    'min_samples_leaf': 20,
    'l2_regularization': 2.0,
    'random_state': SEED,
}

# Exact retained REMOVE policy from Notebook 50.
REMOVE_PROB_THRESHOLD = 0.995
REMOVE_REGION_MIN_MEAN = 0.995
REMOVE_REGION_MIN_PIXEL_FRACTION = 1.00
CONNECTIVITY = 8
MIN_REMOVE_RUN = 2

REPLAY_MAX_ROWS = 4096
REPLAY_MIN_PAGES = 5
REPLAY_MIN_SOURCE_GROUPS = 3

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook59r1_remove_deployment_freeze'
OUT = OUTPUT_ROOT/RUN
BUNDLE = OUT/'bundle'
INPUTS = OUT/'inputs'
TABLES = OUT/'tables'
AUDIT = OUT/'audit'
REPLAY = OUT/'replay'
UPLOAD = OUT/'upload'
for p in [OUT,BUNDLE,INPUTS,TABLES,AUDIT,REPLAY,UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)

## 2. Integrity helpers and completed-run resolution

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, separators=(',', ':')).encode()
    ).hexdigest()

def array_hash(value):
    a = np.ascontiguousarray(value)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path)
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def latest_complete(root, required):
    hits=[]
    root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    unique={str(p.resolve()):p for p in hits}
    return max(unique.values(), key=lambda p:p.stat().st_mtime) if unique else None

NB50_REQUIRED = [
    'selected_models.json',
    'notebook50_verdict.json',
    'experiment_identity.json',
    'inputs/predictor_schema.json',
    'tables/all_m3_oof_predictions_and_decisions.parquet',
]

NB50 = Path(NB50_RUN_OVERRIDE) if NB50_RUN_OVERRIDE else latest_complete(NB50_ROOT, NB50_REQUIRED)
require(NB50 is not None, f'No complete Notebook 50 run found beneath {NB50_ROOT}')
missing=[r for r in NB50_REQUIRED if not (NB50/r).exists()]
require(not missing, f'Notebook 50 run is incomplete: {missing}')
print('Notebook 50:', NB50)

## 3. Verify Notebook 50 identity, selection, verdict and exact lineage

In [ ]:
selected_models=json.loads((NB50/'selected_models.json').read_text())
nb50_verdict=json.loads((NB50/'notebook50_verdict.json').read_text())
nb50_identity=json.loads((NB50/'experiment_identity.json').read_text())
schema=json.loads((NB50/'inputs/predictor_schema.json').read_text())

require(nb50_identity.get('version')==EXPECTED_NB50_VERSION,
        f'Unexpected Notebook 50 version: {nb50_identity.get("version")}')
require(nb50_identity.get('development_only') is True,
        'Notebook 50 identity must be development-only.')
require(nb50_identity.get('promotion_authority') is False,
        'Notebook 50 must not claim promotion authority.')
require(selected_models.get('REMOVE')==EXPECTED_SELECTED_FAMILY,
        f'Unexpected REMOVE family: {selected_models.get("REMOVE")}')
require(nb50_verdict.get('verdict')==EXPECTED_NB50_VERDICT,
        f'Notebook 50 verdict does not authorise the retained REMOVE-only arm: {nb50_verdict.get("verdict")}')
require(nb50_verdict.get('target_passes',{}).get('REMOVE') is True,
        'Notebook 50 REMOVE target did not pass.')
require(nb50_verdict.get('target_passes',{}).get('PROTECT') is False,
        'Notebook 59R1 requires the retained REMOVE-only decision; PROTECT must not be operationalised.')

predictors=(schema.get('ordered_predictors') or schema.get('predictors') or schema.get('ordered_features'))
require(isinstance(predictors,list) and predictors, 'Predictor schema is empty.')
predictors=[str(x) for x in predictors]
require(len(predictors)==len(set(predictors)), 'Predictor schema contains duplicates.')
require(len(predictors)==17, f'Expected 17 frozen predictors; found {len(predictors)}.')

recorded_constants=nb50_identity.get('constants',{})
require(float(recorded_constants.get('remove_threshold'))==REMOVE_PROB_THRESHOLD,
        'Notebook 50 REMOVE threshold mismatch.')
require(int(recorded_constants.get('connectivity'))==CONNECTIVITY,
        'Notebook 50 connectivity mismatch.')
require(int(recorded_constants.get('min_remove_run'))==MIN_REMOVE_RUN,
        'Notebook 50 minimum REMOVE run mismatch.')

# Resolve Notebook 49 from Notebook 50's own frozen input copy/lineage.
# Notebook 50 copied the source tables into its run, but the authoritative training table
# remains the Notebook 49 pixel table identified through Notebook 49R4.
nb49_path_candidates=[]
for path in [NB50/'source_record.json', NB50/'inputs/source_record.json']:
    if path.exists():
        rec=json.loads(path.read_text())
        for key in ['source_notebook49','notebook49_run','source_nb49_run']:
            if rec.get(key): nb49_path_candidates.append(Path(rec[key]))

# Fallback: derive through the current Notebook 49R4 root using its source_record.json.
NB49R4_ROOT=ROOT/'observed_print_classifier_v8_7_notebook49r4'
if not nb49_path_candidates:
    nb49r4=latest_complete(NB49R4_ROOT,[
        'source_record.json','notebook49r4_verdict.json',
        '02_tables/subregion_observable_features.parquet'])
    require(nb49r4 is not None, 'Cannot resolve Notebook 49R4 lineage.')
    r4=json.loads((nb49r4/'source_record.json').read_text())
    if r4.get('source_notebook49'):
        nb49_path_candidates.append(Path(r4['source_notebook49']))

NB49=next((p for p in nb49_path_candidates if (p/'03_truth_unlock/pixel_oracle_gap.parquet').exists()),None)
require(NB49 is not None,
        'Cannot resolve the exact Notebook 49 pixel population. Set the moved source path in the Notebook 49R4 source_record or restore that run.')

print('Selected REMOVE family:',selected_models['REMOVE'])
print('Predictors:',len(predictors))
print('Notebook 49:',NB49)

## 4. Load and prove the exact Notebook 50 development population

In [ ]:
source_pixel_path=NB49/'03_truth_unlock/pixel_oracle_gap.parquet'
source_df=pd.read_parquet(source_pixel_path)
nb50_oof_path=NB50/'tables/all_m3_oof_predictions_and_decisions.parquet'
oof_df=pd.read_parquet(nb50_oof_path)

required_identity={'page','source_group','y','x'}
require(required_identity.issubset(source_df.columns),
        f'Notebook 49 table lacks identities: {sorted(required_identity-set(source_df.columns))}')
require(required_identity.issubset(oof_df.columns),
        f'Notebook 50 table lacks identities: {sorted(required_identity-set(oof_df.columns))}')

label_col=next((c for c in ['truth_class','diagnostic_class','class'] if c in source_df.columns),None)
require(label_col is not None,'No Notebook 50-compatible truth-side class column found.')
source_df[label_col]=source_df[label_col].astype(str).str.upper()

raw_remove_col=f'raw_remove_{EXPECTED_SELECTED_FAMILY}'
require(raw_remove_col in oof_df.columns,
        f'Notebook 50 OOF table lacks preserved raw REMOVE scores: {raw_remove_col}')
require('p_remove' in oof_df.columns,
        'Notebook 50 OOF table lacks calibrated p_remove.')

keys=['page','source_group','y','x']
for frame,name in [(source_df,'Notebook 49 source'),(oof_df,'Notebook 50 OOF')]:
    require(not frame.duplicated(keys).any(), f'{name} contains duplicate identity rows.')

# Merge only the preserved OOF evidence needed for final calibration and audit.
oof_keep=keys+[raw_remove_col,'p_remove','fold']
for optional in ['safe_remove','m5_reason']:
    if optional in oof_df.columns: oof_keep.append(optional)
merged=source_df.merge(oof_df[oof_keep],on=keys,how='inner',validate='one_to_one')
require(len(merged)==len(source_df)==len(oof_df),
        f'Notebook 49/50 population mismatch: source={len(source_df)}, oof={len(oof_df)}, merged={len(merged)}')

missing_predictors=[c for c in predictors if c not in merged.columns]
require(not missing_predictors,f'Frozen predictors missing from training population: {missing_predictors}')

X=merged[predictors].copy()
for c in predictors: X[c]=pd.to_numeric(X[c],errors='coerce')
require(not np.isinf(X.to_numpy(dtype=float)).any(),'Infinite predictor values found.')
require(np.isfinite(pd.to_numeric(merged[raw_remove_col],errors='coerce')).all(),
        'Non-finite preserved OOF raw REMOVE score found.')

y=(merged[label_col].astype(str).str.upper()=='REMOVE').astype(np.uint8)
require(y.nunique()==2,'REMOVE target does not contain both classes.')

population_record={
    'source_notebook49_run':str(NB49),
    'source_notebook50_run':str(NB50),
    'source_pixel_table':str(source_pixel_path),
    'source_pixel_table_sha256':sha256_file(source_pixel_path),
    'notebook50_oof_table':str(nb50_oof_path),
    'notebook50_oof_table_sha256':sha256_file(nb50_oof_path),
    'rows':int(len(merged)),
    'pages':int(merged.page.astype(str).nunique()),
    'source_groups':int(merged.source_group.astype(str).nunique()),
    'remove_positive_rows':int(y.sum()),
    'remove_negative_rows':int((1-y).sum()),
    'predictor_count':len(predictors),
    'ordered_predictors':predictors,
    'identity_columns':keys,
    'label_column':label_col,
    'raw_remove_oof_column':raw_remove_col,
}
atomic_json(AUDIT/'training_population_identity.json',population_record)
merged[keys+['fold',label_col,raw_remove_col,'p_remove']].to_parquet(TABLES/'calibration_evidence.parquet',index=False)
print(json.dumps(population_record,indent=2))

## 5. Fit the selected REMOVE family once on all permitted development rows

In [ ]:
remove_model=Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('model',HistGradientBoostingClassifier(**MODEL_PARAMS)),
])
remove_model.fit(X,y)
require(hasattr(remove_model,'predict_proba'),'Fitted REMOVE model lacks predict_proba.')
require(int(remove_model.n_features_in_)==len(predictors),
        'Fitted REMOVE model feature-count mismatch.')

MODEL_PATH=BUNDLE/'remove_model.joblib'
joblib.dump(remove_model,MODEL_PATH,compress=3)
require(MODEL_PATH.exists() and MODEL_PATH.stat().st_size>0,'REMOVE model was not saved.')

model_record={
    'target':'REMOVE',
    'selected_family':EXPECTED_SELECTED_FAMILY,
    'estimator':'sklearn.pipeline.Pipeline',
    'steps':['impute','model'],
    'imputer':{'class':'SimpleImputer','strategy':'median'},
    'classifier':{'class':'HistGradientBoostingClassifier',**MODEL_PARAMS},
    'training_rows':int(len(X)),
    'positive_rows':int(y.sum()),
    'negative_rows':int((1-y).sum()),
    'predictor_count':len(predictors),
    'model_path':str(MODEL_PATH),
    'model_sha256':sha256_file(MODEL_PATH),
}
atomic_json(AUDIT/'remove_model_fit_record.json',model_record)
print(json.dumps(model_record,indent=2))

## 6. Fit one final calibrator from preserved OOF raw REMOVE scores

In [ ]:
raw_oof=pd.to_numeric(merged[raw_remove_col],errors='raise').to_numpy(np.float64)
final_calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw_oof,y.to_numpy())
calibrated_oof=final_calibrator.predict(raw_oof).astype(np.float64)
require(np.isfinite(calibrated_oof).all(),'Final calibrated OOF probabilities are non-finite.')

CALIBRATOR_PATH=BUNDLE/'remove_calibrator.joblib'
joblib.dump(final_calibrator,CALIBRATOR_PATH,compress=3)
require(CALIBRATOR_PATH.exists() and CALIBRATOR_PATH.stat().st_size>0,'REMOVE calibrator was not saved.')

calibrator_semantics={
    'x_thresholds':final_calibrator.X_thresholds_.tolist(),
    'y_thresholds':final_calibrator.y_thresholds_.tolist(),
}
calibration_record={
    'target':'REMOVE',
    'class':'IsotonicRegression',
    'out_of_bounds':'clip',
    'fit_source':'preserved Notebook 50 source-group OOF raw REMOVE scores and REMOVE truth',
    'in_sample_full_fit_scores_used':False,
    'rows':int(len(raw_oof)),
    'positive_rows':int(y.sum()),
    'negative_rows':int((1-y).sum()),
    'raw_score_sha256':array_hash(raw_oof),
    'truth_sha256':array_hash(y.to_numpy(np.uint8)),
    'calibrated_oof_sha256':array_hash(calibrated_oof),
    'semantic_sha256':object_hash(calibrator_semantics),
    'calibrator_path':str(CALIBRATOR_PATH),
    'calibrator_sha256':sha256_file(CALIBRATOR_PATH),
}
atomic_json(AUDIT/'remove_calibrator_fit_record.json',calibration_record)
pd.DataFrame({'raw_remove_oof':raw_oof,'calibrated_remove_oof':calibrated_oof,'remove_truth':y}).to_parquet(TABLES/'final_calibrator_oof_fit.parquet',index=False)
print(json.dumps(calibration_record,indent=2))

## 7. Freeze schema, policy, reason codes and compatibility identity

In [ ]:
SCHEMA_PATH=BUNDLE/'predictor_schema.json'
POLICY_PATH=BUNDLE/'policy.json'
REASON_PATH=BUNDLE/'reason_code_schema.json'

schema_out={
    'schema_version':'notebook59r1-remove-deployment-v1',
    'ordered_predictors':predictors,
    'predictor_count':len(predictors),
    'dtype':'numeric; pandas coercion; infinities prohibited; missing values handled only by frozen median imputer',
    'source_notebook50_schema_sha256':sha256_file(NB50/'inputs/predictor_schema.json'),
}
atomic_json(SCHEMA_PATH,schema_out)

reason_codes={
    'FALLBACK_M3':0,
    'SAFE_REMOVE':1,
    'INVALID_OOD_FALLBACK_M3':4,
}
atomic_json(REASON_PATH,reason_codes)

compatibility_payload={
    'model_family':EXPECTED_SELECTED_FAMILY,
    'model_parameters':MODEL_PARAMS,
    'ordered_predictors':predictors,
    'target':'REMOVE',
    'threshold':REMOVE_PROB_THRESHOLD,
    'region_min_mean':REMOVE_REGION_MIN_MEAN,
    'region_min_pixel_fraction':REMOVE_REGION_MIN_PIXEL_FRACTION,
    'connectivity':CONNECTIVITY,
    'minimum_run':MIN_REMOVE_RUN,
    'fallback':'exact M3',
    'protect_enabled':False,
}
compatibility_id='v8.8-remove-'+object_hash(compatibility_payload)[:20]
policy_id='v8.8-remove-policy-'+object_hash({
    **compatibility_payload,
    'model_sha256':sha256_file(MODEL_PATH),
    'calibrator_sha256':sha256_file(CALIBRATOR_PATH),
    'schema_sha256':sha256_file(SCHEMA_PATH),
})[:20]

policy={
    'policy_id':policy_id,
    'compatibility_id':compatibility_id,
    'target':'REMOVE',
    'selected_family':EXPECTED_SELECTED_FAMILY,
    'remove_probability_threshold':REMOVE_PROB_THRESHOLD,
    'remove_region_minimum_mean_probability':REMOVE_REGION_MIN_MEAN,
    'remove_region_minimum_pixel_fraction_above_threshold':REMOVE_REGION_MIN_PIXEL_FRACTION,
    'connectivity':CONNECTIVITY,
    'minimum_remove_run':MIN_REMOVE_RUN,
    'page_global_region_filter':True,
    'safe_protect_enabled':False,
    'unsupported_invalid_uncertain_fallback':'EXACT_M3',
    'model_file':'remove_model.joblib',
    'model_sha256':sha256_file(MODEL_PATH),
    'calibrator_file':'remove_calibrator.joblib',
    'calibrator_sha256':sha256_file(CALIBRATOR_PATH),
    'predictor_schema_file':'predictor_schema.json',
    'predictor_schema_sha256':sha256_file(SCHEMA_PATH),
    'reason_code_schema_file':'reason_code_schema.json',
    'reason_code_schema_sha256':sha256_file(REASON_PATH),
    'source_notebook50_run':str(NB50),
    'source_notebook50_identity_sha256':sha256_file(NB50/'experiment_identity.json'),
    'source_notebook50_verdict_sha256':sha256_file(NB50/'notebook50_verdict.json'),
    'source_notebook50_selected_models_sha256':sha256_file(NB50/'selected_models.json'),
    'scientific_selection_evidence':'Notebook 50 source-group OOF REMOVE evidence',
    'deployment_fit_role':'all-permitted-development fit of the already-selected family',
    'promotion_authority':False,
}
atomic_json(POLICY_PATH,policy)
print(json.dumps(policy,indent=2))

## 8. Deterministic reload and frozen multi-page replay

In [ ]:
reloaded_model=joblib.load(MODEL_PATH)
reloaded_calibrator=joblib.load(CALIBRATOR_PATH)

# Deterministic, identity-sorted, source-diverse replay sample.
ordered=merged.sort_values(['source_group','page','y','x']).reset_index(drop=True)
groups=ordered.source_group.astype(str).drop_duplicates().tolist()
require(len(groups)>=REPLAY_MIN_SOURCE_GROUPS,
        f'Need at least {REPLAY_MIN_SOURCE_GROUPS} source groups for replay.')
selected_groups=groups[:max(REPLAY_MIN_SOURCE_GROUPS,min(len(groups),5))]
parts=[]
per_group=max(1,REPLAY_MAX_ROWS//len(selected_groups))
for group in selected_groups:
    q=ordered[ordered.source_group.astype(str)==group]
    if len(q):
        step=max(1,len(q)//per_group)
        parts.append(q.iloc[::step].head(per_group))
replay_df=pd.concat(parts,ignore_index=True).head(REPLAY_MAX_ROWS)
if replay_df.page.astype(str).nunique()<REPLAY_MIN_PAGES:
    replay_df=ordered.groupby('page',sort=True,group_keys=False).head(max(1,REPLAY_MAX_ROWS//max(ordered.page.nunique(),1))).head(REPLAY_MAX_ROWS)
require(replay_df.page.astype(str).nunique()>=REPLAY_MIN_PAGES,
        f'Replay sample has fewer than {REPLAY_MIN_PAGES} pages.')

replay_X=replay_df[predictors].copy()
for c in predictors: replay_X[c]=pd.to_numeric(replay_X[c],errors='coerce')
raw_a=remove_model.predict_proba(replay_X)[:,1].astype(np.float64)
raw_b=reloaded_model.predict_proba(replay_X)[:,1].astype(np.float64)
prob_a=final_calibrator.predict(raw_a).astype(np.float64)
prob_b=reloaded_calibrator.predict(raw_b).astype(np.float64)

require(np.array_equal(raw_a,raw_b),'Saved REMOVE model failed exact reload replay.')
require(np.array_equal(prob_a,prob_b),'Saved REMOVE calibrator failed exact reload replay.')
require(np.isfinite(raw_b).all() and np.isfinite(prob_b).all(),'Replay scores are non-finite.')

replay_out=replay_df[['page','source_group','y','x']].copy()
replay_out['raw_remove_probability']=raw_b
replay_out['calibrated_remove_probability']=prob_b
replay_out['provisional_remove']=prob_b>=REMOVE_PROB_THRESHOLD
REPLAY_TABLE=REPLAY/'replay_expected.parquet'
replay_out.to_parquet(REPLAY_TABLE,index=False)

replay_record={
    'passed':True,
    'policy_id':policy_id,
    'compatibility_id':compatibility_id,
    'rows':int(len(replay_out)),
    'pages':int(replay_out.page.astype(str).nunique()),
    'source_groups':int(replay_out.source_group.astype(str).nunique()),
    'model_reload_exact':True,
    'calibrator_reload_exact':True,
    'raw_probability_sha256':array_hash(raw_b),
    'calibrated_probability_sha256':array_hash(prob_b),
    'provisional_decision_sha256':array_hash(replay_out.provisional_remove.to_numpy(np.uint8)),
    'replay_table':'replay/replay_expected.parquet',
    'replay_table_sha256':sha256_file(REPLAY_TABLE),
    'note':'Region coherence is applied page-globally by the runtime after all candidate rows for a page are scored.',
}
atomic_json(REPLAY/'replay_test.json',replay_record)
print(json.dumps(replay_record,indent=2))

## 9. Write runtime contract for Notebooks 60R1/60R2

In [ ]:
runtime_contract={
    'contract_version':'notebook59r1-remove-runtime-v1',
    'policy_id':policy_id,
    'compatibility_id':compatibility_id,
    'bundle_root':str(BUNDLE),
    'required_files':[
        'remove_model.joblib','remove_calibrator.joblib','predictor_schema.json',
        'policy.json','reason_code_schema.json','freeze_manifest.json'
    ],
    'scoring_sequence':[
        'construct the exact 17 frozen predictors in predictor_schema.json order',
        'coerce predictors to numeric and reject infinities',
        'raw_remove = remove_model.predict_proba(X)[:,1]',
        'p_remove = remove_calibrator.predict(raw_remove)',
        'provisional REMOVE iff p_remove >= 0.995',
        'assemble provisional decisions for the complete page',
        'group provisional decisions using 8-connectivity',
        'SAFE_REMOVE only when region size >= 2, region mean >= 0.995, and every region pixel >= 0.995',
        'all other rows remain exact M3 fallback',
    ],
    'input_failure_behaviour':'fail closed to exact M3 and write the failure reason',
    'safe_protect_enabled':False,
    'truth_inputs_permitted_at_runtime':False,
    'clean_page_required_at_runtime':False,
}
atomic_json(BUNDLE/'runtime_contract.json',runtime_contract)

## 10. Immutable freeze manifest, environment and final upload package

In [ ]:
environment={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'python':sys.version,
    'platform':platform.platform(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'scikit_learn':sklearn.__version__,
    'joblib':joblib.__version__,
}
atomic_json(OUT/'software_environment.json',environment)

# Manifest all objects except the manifest itself, then seal the manifest.
manifest_rows=[]
for p in sorted(BUNDLE.iterdir()):
    if p.is_file() and p.name!='freeze_manifest.json':
        manifest_rows.append({
            'relative_path':p.name,
            'bytes':p.stat().st_size,
            'sha256':sha256_file(p),
            'required_at_runtime':p.name in {
                'remove_model.joblib','remove_calibrator.joblib','predictor_schema.json',
                'policy.json','reason_code_schema.json','runtime_contract.json'
            },
        })
manifest={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.8-notebook59r1-remove-deployment-freeze',
    'policy_id':policy_id,
    'compatibility_id':compatibility_id,
    'source_notebook50_run':str(NB50),
    'source_notebook49_run':str(NB49),
    'selected_target':'REMOVE',
    'selected_family':EXPECTED_SELECTED_FAMILY,
    'protect_included':False,
    'training_population':population_record,
    'model_fit_record':model_record,
    'calibration_fit_record':calibration_record,
    'replay':replay_record,
    'software_environment':environment,
    'objects':manifest_rows,
    'deployment_bundle_complete':True,
    'independent_promotion_authority':False,
}
atomic_json(BUNDLE/'freeze_manifest.json',manifest)

# Verify every sealed hash after writing the manifest.
for row in manifest_rows:
    p=BUNDLE/row['relative_path']
    require(p.exists() and p.stat().st_size==row['bytes'] and sha256_file(p)==row['sha256'],
            f'Post-freeze hash failure: {p}')

# Copy review/audit essentials. Heavy training tables stay in the durable run.
review_files=[
    BUNDLE/'remove_model.joblib',BUNDLE/'remove_calibrator.joblib',
    BUNDLE/'predictor_schema.json',BUNDLE/'policy.json',
    BUNDLE/'reason_code_schema.json',BUNDLE/'runtime_contract.json',BUNDLE/'freeze_manifest.json',
    REPLAY/'replay_test.json',AUDIT/'training_population_identity.json',
    AUDIT/'remove_model_fit_record.json',AUDIT/'remove_calibrator_fit_record.json',
    OUT/'software_environment.json',
]
for p in review_files:
    require(p.exists(),f'Missing upload artefact: {p}')
    shutil.copy2(p,UPLOAD/p.name)

upload_rows=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():
        upload_rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(upload_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)

summary_lines=[
    'NOTEBOOK 59R1 - V8.8 REMOVE DEPLOYMENT FIT AND FREEZE','',
    f'Run: {OUT}',
    f'Source Notebook 50: {NB50}',
    f'Source Notebook 49: {NB49}',
    f'Policy ID: {policy_id}',
    f'Compatibility ID: {compatibility_id}',
    f'Training rows: {len(merged)}',
    f'Pages: {merged.page.astype(str).nunique()}',
    f'Source groups: {merged.source_group.astype(str).nunique()}',
    f'Predictors: {len(predictors)}',
    'Selected target/family: REMOVE / local_nonlinear',
    'Final calibration source: preserved Notebook 50 OOF raw REMOVE scores and truth',
    f'Replay rows/pages/groups: {replay_record["rows"]} / {replay_record["pages"]} / {replay_record["source_groups"]}',
    'Exact model reload replay: PASS',
    'Exact calibrator reload replay: PASS',
    'SAFE_PROTECT included: False',
    'Fallback: exact M3',
    '',
    'This notebook creates the missing fresh-page scoring bundle. It does not replace Notebook 50 OOF evidence, claim independent validation, or authorise production promotion.'
]
(OUT/'executive_summary.txt').write_text('\n'.join(summary_lines))
shutil.copy2(OUT/'executive_summary.txt',UPLOAD/'executive_summary.txt')
(UPLOAD/'README.txt').write_text('Start with executive_summary.txt, freeze_manifest.json, policy.json, runtime_contract.json and replay_test.json. Notebook 59R1 creates the missing REMOVE deployment scorer; it does not authorise production promotion.')
shutil.make_archive(str(OUT/'notebook59r1_upload_bundle'),'zip',UPLOAD)

print('\n'.join(summary_lines))
print('Bundle:',BUNDLE)
print('Upload ZIP:',OUT/'notebook59r1_upload_bundle.zip')

## Outputs to inspect first

1. `executive_summary.txt`
2. `bundle/freeze_manifest.json`
3. `bundle/policy.json`
4. `bundle/runtime_contract.json`
5. `bundle/predictor_schema.json`
6. `replay/replay_test.json`
7. `audit/training_population_identity.json`
8. `audit/remove_model_fit_record.json`
9. `audit/remove_calibrator_fit_record.json`
10. `notebook59r1_upload_bundle.zip`

### Handoff to 60R1/60R2

Notebooks 60R1/60R2 must load only a Notebook 59R1 bundle whose manifest, model, calibrator, schema, policy and runtime-contract hashes verify. They must not search Notebook 50 for a model, average Notebook 50 fold calibrators, reconstruct a missing estimator, enable SAFE_PROTECT, or change the 0.995/region-coherence rules.